In [1]:
import ConnectionConfig as cc
from delta import DeltaTable
from pyspark.sql.functions import col,explode, from_unixtime, unix_timestamp
cc.setupEnvironment()

In [2]:
spark = cc.startLocalCluster("factRit")
spark.getActiveSession()

In [3]:
cc.set_connectionProfile("VeloDB")

df_operational_rides = spark.read \
    .format("jdbc") \
    .option("url", cc.create_jdbc()) \
    .option("driver" , cc.get_Property("driver")) \
    .option("dbtable", "rides") \
    .option("user", cc.get_Property("username")) \
    .option("password", cc.get_Property("password")) \
    .option("partitionColumn", "rideId") \
    .option("numPartitions", 4) \
    .option("lowerBound", 0) \
    .option("upperBound", 1000) \
    .load()

# Load the JSON files from the weather folder
df_weather = spark.read.json("./weather/", multiLine=True)

df_weather = df_weather.withColumn("weather_time", from_unixtime(col("dt")))

# Flatten the DataFrame
df_weather_flat = df_weather.select(
    col("zipCode"),
    col("coord.lon"),
    col("coord.lat"),
    col("main.temp"),
    col("main.pressure"),
    col("main.humidity"),
    col("wind.speed"),
    col("clouds.all"),
    col("weather_time"),
    col("sys.country"),
    col("timezone"),
    col("name"),
    explode(col("weather")).alias("weather")  # Exploding the weather array
)

# Select specific fields from the exploded weather structure
df_weather_flat = df_weather_flat.select(
    col("zipCode"),
    col("lon"),
    col("lat"),
    col("temp"),
    col("pressure"),
    col("humidity"),
    col("speed"),
    col("all").alias("cloudiness"),
    col("weather_time"),  # This is still the original timestamp
    col("country"),
    col("timezone"),
    col("name"),
    col("weather.id").alias("weather_id"),
    col("weather.main").alias("weather_main"),
    col("weather.description").alias("weather_description")
)

# Create a view for SQL queries
df_weather_flat.createOrReplaceTempView("weather_source")

# Show the weather data to confirm it loaded correctly
df_weather_flat.show(truncate=False)

df_operational_stations = spark.read \
    .format("jdbc") \
    .option("url", cc.create_jdbc()) \
    .option("driver" , cc.get_Property("driver")) \
    .option("dbtable", "stations") \
    .option("user", cc.get_Property("username")) \
    .option("password", cc.get_Property("password")) \
    .option("partitionColumn", "stationId") \
    .option("numPartitions", 4) \
    .option("lowerBound", 0) \
    .option("upperBound", 1000) \
    .load()
    

+-------+------+-------+-----+--------+--------+-----+----------+-------------------+-------+--------+--------------------+----------+------------+-------------------+
|zipCode|lon   |lat    |temp |pressure|humidity|speed|cloudiness|weather_time       |country|timezone|name                |weather_id|weather_main|weather_description|
+-------+------+-------+-----+--------+--------+-----+----------+-------------------+-------+--------+--------------------+----------+------------+-------------------+
|2140   |4.4167|51.2167|15.38|1000    |90      |1.79 |100       |2024-10-07 20:58:17|BE     |7200    |Antwerpen Borgerhout|801       |Clouds      |few clouds         |
|2140   |4.4167|51.2167|15.38|1000    |90      |1.79 |100       |2024-10-07 20:58:17|BE     |7200    |Antwerpen Borgerhout|800       |Clear       |clear sky          |
|2140   |4.4167|51.2167|15.38|1000    |90      |1.79 |100       |2024-10-07 20:58:17|BE     |7200    |Antwerpen Borgerhout|500       |Rain        |light rain   

#### 2 MAKE DIMENSION TABLES AVAILABLE AS VIEWS


In [4]:
dimDate = spark.read.format("delta").load("spark-warehouse/dimDate")
dim_Customer = spark.read.format("delta").load("spark-warehouse/dimCustomer")
dim_locks = spark.read.format("delta").load("spark-warehouse/dimLocks")

dim_Customer.createOrReplaceTempView("dimCustomer")
dim_locks.createOrReplaceTempView("dimLocks")
dimDate.createOrReplaceTempView("dimDate")

In [5]:
df_operational_rides.createOrReplaceTempView("rides_source")
df_operational_stations.createOrReplaceTempView("stations_source")

In [6]:
# todo scd doen ## uuid neemt veel ruimte 
fact_rides = spark.sql("""
    SELECT  
        r.rideId AS rideId,
        s1.lockSK AS startLockSK,
        s2.lockSK AS endLockSK,
        r.starttime AS starttime,
        r.endtime AS endtime,
        c.customer_id AS customer_id,
        c.customer_SK AS customerSK,
        w.zipCode AS zipCode,
        d.dateSK AS dateSK,
        w.weather_main AS weatherCondition,  -- Extracting the main weather condition
        w.weather_description AS weatherDescription  -- Extracting the description
    FROM 
        rides_source AS r
    LEFT OUTER JOIN 
        dimLocks AS s1 ON r.startlockId = s1.lockId
    LEFT OUTER JOIN 
        dimLocks AS s2 ON r.endlockId = s2.lockId
    LEFT OUTER JOIN 
        dimCustomer AS c ON r.subscriptionid = c.customer_id
    LEFT OUTER JOIN 
        dimDate AS d ON date_format(r.starttime, 'yyyy-MM-dd') = d.calendarDate
    LEFT OUTER JOIN 
        weather_source AS w ON s1.zipCode = w.zipCode 
    WHERE 
        s1.lockId IS NOT NULL AND 
        s2.lockId IS NOT NULL AND 
        c.customer_id IS NOT NULL
""")

# Show the result
fact_rides.show(20)




+------+-----------+---------+-------------------+-------------------+-----------+-------+------+----------------+------------------+
|rideId|startLockSK|endLockSK|          starttime|            endtime|customer_id|zipCode|dateSK|weatherCondition|weatherDescription|
+------+-----------+---------+-------------------+-------------------+-----------+-------+------+----------------+------------------+
|     1|        992|      681|2019-09-22 08:46:43|2019-09-22 09:01:36|      13296|   2140|  3916|            Rain|        light rain|
|     1|        992|      681|2019-09-22 08:46:43|2019-09-22 09:01:36|      13296|   2140|  3916|           Clear|         clear sky|
|     1|        992|      681|2019-09-22 08:46:43|2019-09-22 09:01:36|      13296|   2140|  3916|          Clouds|        few clouds|
|     3|       4147|     2270|2019-09-22 08:27:38|2019-09-22 08:30:25|      25722|   2000|  3916|            Rain|        light rain|
|     3|       4147|     2270|2019-09-22 08:27:38|2019-09-22 0

In [7]:
delta_table_path = "spark-warehouse/factRide"
fact_rides.write.format("delta").mode("overwrite").save(delta_table_path)

Py4JJavaError: An error occurred while calling o185.save.
: org.apache.spark.SparkException: Job 29 cancelled because SparkContext was shut down
	at org.apache.spark.scheduler.DAGScheduler.$anonfun$cleanUpAfterSchedulerStop$1(DAGScheduler.scala:1253)
	at org.apache.spark.scheduler.DAGScheduler.$anonfun$cleanUpAfterSchedulerStop$1$adapted(DAGScheduler.scala:1251)
	at scala.collection.mutable.HashSet.foreach(HashSet.scala:79)
	at org.apache.spark.scheduler.DAGScheduler.cleanUpAfterSchedulerStop(DAGScheduler.scala:1251)
	at org.apache.spark.scheduler.DAGSchedulerEventProcessLoop.onStop(DAGScheduler.scala:3087)
	at org.apache.spark.util.EventLoop.stop(EventLoop.scala:84)
	at org.apache.spark.scheduler.DAGScheduler.$anonfun$stop$3(DAGScheduler.scala:2973)
	at org.apache.spark.util.Utils$.tryLogNonFatalError(Utils.scala:1375)
	at org.apache.spark.scheduler.DAGScheduler.stop(DAGScheduler.scala:2973)
	at org.apache.spark.SparkContext.$anonfun$stop$12(SparkContext.scala:2258)
	at org

In [ ]:
spark.stop()